In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)

In [5]:
df = pd.read_csv("../EDA/final_model.csv")
df["Start Date"] = pd.to_datetime(df["Start Date"])
df = df.sort_values("Start Date").reset_index(drop=True)

In [6]:
df.head()

,Runs,Pos,Inns,Ground,Start Date,NotOut,Rolling5,Rolling10,Opposition_v Australia,Opposition_v Bangladesh,Opposition_v England,Opposition_v Ireland,Opposition_v Netherlands,Opposition_v New Zealand,Opposition_v Pakistan,Opposition_v South Africa,Opposition_v Sri Lanka,Opposition_v U.A.E.,Opposition_v West Indies,Opposition_v Zimbabwe
0,37,2,2,Dambulla,2008-08-20,0,12.000000,12.000000,0,0,0,0,0,0,0,0,1,0,0,0
1,25,1,1,Colombo (RPS),2008-08-24,0,24.500000,24.500000,0,0,0,0,0,0,0,0,1,0,0,0
2,54,1,1,Colombo (RPS),2008-08-27,0,24.666667,24.666667,0,0,0,0,0,0,0,0,1,0,0,0
3,31,1,2,Colombo (RPS),2008-08-29,0,32.000000,32.000000,0,0,0,0,0,0,0,0,1,0,0,0
4,2,7,1,Colombo (RPS),2009-09-14,1,31.800000,31.800000,0,0,0,0,0,0,0,0,1,0,0,0


In [7]:
opposition_cols = [
    col for col in df.columns
    if col.startswith("Opposition_")
]

In [8]:
opposition_cols

['Opposition_v Australia',
 'Opposition_v Bangladesh',
 'Opposition_v England',
 'Opposition_v Ireland',
 'Opposition_v Netherlands',
 'Opposition_v New Zealand',
 'Opposition_v Pakistan',
 'Opposition_v South Africa',
 'Opposition_v Sri Lanka',
 'Opposition_v U.A.E.',
 'Opposition_v West Indies',
 'Opposition_v Zimbabwe']

In [9]:
numeric_feats = ["Rolling5","Rolling10"]
categorical_feats = ["Ground"]

feats = numeric_feats + categorical_feats + opposition_cols

In [10]:
feats

['Rolling5',
 'Rolling10',
 'Ground',
 'Opposition_v Australia',
 'Opposition_v Bangladesh',
 'Opposition_v England',
 'Opposition_v Ireland',
 'Opposition_v Netherlands',
 'Opposition_v New Zealand',
 'Opposition_v Pakistan',
 'Opposition_v South Africa',
 'Opposition_v Sri Lanka',
 'Opposition_v U.A.E.',
 'Opposition_v West Indies',
 'Opposition_v Zimbabwe']

In [11]:
X = df[feats]
y = df["Runs"]

In [13]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

n,train_end,val_end

(298, 208, 253)

In [14]:
X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

In [15]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        numeric_feats,
    ),
    (
        "ground",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]),
        categorical_feats,
    ),
    ("opposition", "passthrough", opposition_cols),
])

In [16]:
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]

In [17]:
validation_results = []

for alpha in alpha_values:
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("lasso", Lasso(alpha=alpha, max_iter=20000)),
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    validation_results.append({
        "alpha": alpha,
        "MAE": mean_absolute_error(y_val, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_val, predictions)),
        "R2": r2_score(y_val, predictions),
    })

In [18]:
validation_df = pd.DataFrame(validation_results)
print(validation_df.sort_values("MAE").to_string(index=False))

  alpha       MAE      RMSE        R2
100.000 31.498825 36.587773 -0.003250
 10.000 31.498825 36.587773 -0.003250
  1.000 32.076727 37.279363 -0.041535
  0.100 35.500713 43.286147 -0.404219
  0.010 37.446306 45.711204 -0.565966
  0.001 37.732654 46.037655 -0.588413


In [20]:
best_alpha = validation_df.loc[
    validation_df["MAE"].idxmin(), "alpha"
]
best_alpha

np.float64(10.0)

In [23]:
ridge_trainval_X = X.iloc[:val_end]
ridge_trainval_y = y.iloc[:val_end]

In [24]:
lasso_model = Pipeline([
    ("preprocessor", preprocessor),
    ("lasso", Lasso(alpha=best_alpha, max_iter=20000)),
])

In [25]:
lasso_model.fit(ridge_trainval_X, ridge_trainval_y)
test_predictions = lasso_model.predict(X_test)

In [27]:
lasso_results = {
    "Model": "Lasso Regression",
    "Alpha": best_alpha,
    "MAE": mean_absolute_error(y_test, test_predictions),
    "RMSE": np.sqrt(mean_squared_error(y_test, test_predictions)),
    "R2": r2_score(y_test, test_predictions),
}

print("Test Results:")
print(pd.Series(lasso_results))

Test Results:
Model    Lasso Regression
Alpha                10.0
MAE             40.303645
RMSE            46.411925
R2              -0.014414
dtype: object


In [28]:
lasso = lasso_model.named_steps["lasso"]

print("Total coefficients:", len(lasso.coef_))
print("Zero coefficients:", np.sum(np.isclose(lasso.coef_, 0)))
print(
    "Non-zero coefficients:",
    np.sum(~np.isclose(lasso.coef_, 0))
)

Total coefficients: 75
Zero coefficients: 75
Non-zero coefficients: 0
